# Mikado Synthetic Data Generator

Generates synthetic Mikado training images using Blender's rigid-body physics.
Sticks are dropped into a pile, rendered from above, and YOLO-OBB labels are exported automatically.

**Runtime → Change runtime type → T4 GPU** (optional — speeds up Cycles rendering)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURE THESE
# ---------------------------------------------------------------------------
DRIVE_ROOT = '/content/drive/MyDrive/Data/mikado-judge'

# Where to save the generated dataset
OUTPUT_DIR = f'{DRIVE_ROOT}/synthetic'

# Number of images to generate
COUNT = 200

# Random seed (0 = different each run)
SEED = 0

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')
print(f'Count:  {COUNT}')

## 2. Install Blender

In [ ]:
# Install Blender (headless)
!apt-get install -y blender > /dev/null 2>&1
!blender --version

In [ ]:
# Install pyyaml inside Blender's bundled Python
import subprocess, sys
blender_python = subprocess.check_output(
    ['blender', '--background', '--python-expr',
     'import sys; print(sys.executable)'],
    text=True
).strip().split('\n')[-1]
print(f'Blender Python: {blender_python}')
!{blender_python} -m pip install pyyaml --quiet

## 3. Clone repository

In [ ]:
REPO_DIR = '/content/mikado-blender'
REPO_URL = 'https://github.com/spyszkowski/mikado-blender.git'

import os
if os.path.exists(REPO_DIR):
    print('Repository exists, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning...')
    !git clone {REPO_URL} {REPO_DIR}

print('Done.')

## 4. Generate synthetic images

In [ ]:
!blender --background --python {REPO_DIR}/scripts/generate.py -- \
    --count {COUNT} \
    --output {OUTPUT_DIR} \
    --config {REPO_DIR}/configs \
    --seed {SEED}

## 5. Verify output

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image, ImageDraw
import random

images = sorted(Path(OUTPUT_DIR, 'images').glob('*.png'))
labels = sorted(Path(OUTPUT_DIR, 'labels').glob('*.txt'))
print(f'Generated: {len(images)} images, {len(labels)} label files')

# Count total annotations
total_sticks = sum(len(l.read_text().strip().splitlines()) for l in labels if l.stat().st_size > 0)
print(f'Total stick annotations: {total_sticks}')
if images:
    print(f'Avg sticks per image: {total_sticks / len(images):.1f}')

In [ ]:
# Show 4 sample images with OBB overlays
CLASS_COLORS = {
    0: (180, 0, 180),   # mikado — purple
    1: (30, 80, 230),   # blue
    2: (230, 30, 30),   # red
    3: (230, 200, 30),  # yellow
    4: (30, 180, 60),   # green
}

samples = random.sample(list(zip(images, labels)), min(4, len(images)))
for img_path, lbl_path in samples:
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    draw = ImageDraw.Draw(img)
    for line in lbl_path.read_text().strip().splitlines():
        parts = list(map(float, line.split()))
        cid = int(parts[0])
        pts = [(parts[i]*w, parts[i+1]*h) for i in range(1, 9, 2)]
        color = CLASS_COLORS.get(cid, (255, 255, 255))
        draw.polygon(pts, outline=color)
    img = img.resize((min(w, 1200), int(h * min(w, 1200) / w)))
    print(img_path.name)
    display(img)

## 6. Package for use with mikado-judge

In [ ]:
# Create a tar.gz archive ready to merge with the real dataset
import tarfile
from pathlib import Path

archive_path = f'{DRIVE_ROOT}/synthetic.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(OUTPUT_DIR, arcname='synthetic')
print(f'Archived → {archive_path}')
print('Download this and merge with your real dataset before re-running prepare_dataset.py')